# Floci masterclass: live walkthrough

Floci is a local, open source AWS emulator, similar to LocalStack. It
answers real boto3 calls at `http://floci:4566`, using the same API as
AWS. You don't need an AWS account, and nothing costs money.

This notebook runs two demos. In the first, you upload a file to S3 and
watch it trigger a Lambda function, with no server, cron job, or polling
loop involved. Four separate pieces of AWS configuration have to agree
before that works, and you'll build each one by hand so you can see where
it fits. In the second, you create an AWS user with no permissions and
watch AWS refuse every request it makes, then grant one specific
permission and watch the same request succeed. Floci enforces that
exactly like real AWS, once you turn enforcement on.

The `demos/` folder has scripts that do the same thing in one shot:
`deploy.py`, `trigger_and_wait.py`, `run_demo.py`. Those are what
`make demo1`, `make demo2`, and CI actually run. This notebook unpacks
the same AWS calls into one cell per step, with an explanation of why
each step exists, so you can run and discuss them live.

Before you start, run `make up` in a terminal. That starts Floci and this
notebook's kernel. You only need to do it once per session.

In [ ]:
import os
from pathlib import Path

# Euporie and `jupyter execute` may start the kernel in either the repo
# root or this notebook's own directory — normalize to the repo root so
# every relative path below (demos/...) resolves the same way either way.
if not Path("demos").exists():
    os.chdir(Path.cwd().parent)

Path.cwd()

## Connect to Floci

Run the next cell to check that Floci is up and answering requests. It
should print a version string such as `2.1.0`. If it raises a connection
error, open a terminal, run `make up`, then run this cell again.

In [ ]:
import json
import time
import urllib.parse
import urllib.request

import boto3
from botocore.exceptions import ClientError

ENDPOINT = os.environ.get("FLOCI_ENDPOINT", "http://floci:4566")

health = urllib.request.urlopen(f"{ENDPOINT}/_floci/health")
json.loads(health.read())["version"]

---
## Demo 1: an upload triggers a Lambda

```
┌───────────────┐           ┌─────────┐         ┌────────────────┐
│ in/scores.csv │           │ predict │         │ out/scores.csv │
│               │ S3 event> │         │ writes> │                │
│ score,pred?   │           │         │         │ score,pred     │
└───────────────┘           └─────────┘         └────────────────┘
```

Here's the story: someone uploads a batch of scores to S3, and a Lambda
function called `predict` reads it, adds a prediction column, and writes
the result back to S3. No server runs in between. No cron job checks for
new files. The upload itself is what starts the Lambda.

For that to work, four separate pieces of AWS configuration have to
agree:

1. A trust policy tells IAM which service may run as `predict`. Without
   it, Lambda can't even start.
2. An execution policy tells IAM what `predict` may do once it's
   running: read from `in/`, write to `out/`. Without it, the function
   starts but fails with `AccessDenied`.
3. An invoke permission tells Lambda that S3 is allowed to call it.
   Without it, S3 refuses to register a notification for this function
   at all, before any file is ever uploaded.
4. An event notification tells S3 which uploads should trigger
   `predict`. Without it, nothing happens when you upload a file, even
   though everything else is wired correctly.

We'll build these in that order, 1 to 4, which is also the natural order
to set them up in. But once a file actually lands, they fire in the
opposite order: S3 checks its notification rule first, then checks
whether it's allowed to invoke Lambda, then Lambda checks whether it's
allowed to assume its role, and only then does the function run and hit
the execution policy. Keep that reversal in mind as you read the code
below.

### Read the handler before wiring anything

Look at the function `predict` is going to run, before building any
infrastructure around it. The handler expects three things from its
environment: an S3 event describing what file was uploaded (steps 3 and
4 make that arrive), permission to read that file, and permission to
write the result somewhere (step 2 grants both).

In [ ]:
print(Path("demos/01-s3-lambda-notification/lambda_predict/handler.py").read_text())

Two of these are easy to mix up, because they both involve permissions,
but they answer different questions. The trust policy (step 1) controls
who may *become* `predict`: who can assume `lambda-exec-role` and run
code as it. The execution policy (step 2) controls what `predict` may
*do* once it's running, scoped to that same role. The invoke permission
(step 3) is different again: it isn't about the role at all. It's a
permission attached directly to the Lambda function, stating that the S3
service specifically is allowed to call it.

```
┌────────┐          ┌───────────┐      ┌──────────────┐
│ Lambda │ assumes> │ exec role │ r/w> │ S3 in/, out/ │
└────────┘          └───────────┘      └──────────────┘
   step 1                 step 2

┌────────────┐          ┌────────────────┐
│ S3 service │ invokes> │ predict Lambda │
└────────────┘          └────────────────┘
     step 3 (a permission on the function itself)
```

The cells below build steps 1 to 4, in order.

In [ ]:
iam = boto3.client("iam", endpoint_url=ENDPOINT)
lam = boto3.client("lambda", endpoint_url=ENDPOINT)
s3 = boto3.client("s3", endpoint_url=ENDPOINT)

ROLE_NAME = "lambda-exec-role"
FUNC_NAME = "predict"
BUCKET = "demo"

### Step 1: the trust policy

This policy tells IAM which service may assume `lambda-exec-role`. Here,
only `lambda.amazonaws.com` can. No other identity, human or automated,
can act as `predict` using this role.

Run the cell below. It creates the role, or reuses it if this notebook
has already run once. Expect an ARN back, similar to
`arn:aws:iam::000000000000:role/lambda-exec-role`.

In [ ]:
trust_policy = {
    "Version": "2012-10-17",
    "Statement": [
        {
            "Effect": "Allow",
            "Principal": {"Service": "lambda.amazonaws.com"},
            "Action": "sts:AssumeRole",
        }
    ],
}

try:
    role = iam.create_role(
        RoleName=ROLE_NAME, AssumeRolePolicyDocument=json.dumps(trust_policy)
    )
    role_arn = role["Role"]["Arn"]
except iam.exceptions.EntityAlreadyExistsException:
    role_arn = iam.get_role(RoleName=ROLE_NAME)["Role"]["Arn"]

role_arn

### Step 2: the execution policy

This is what `predict` can actually do once IAM lets it run: read
objects under `demo/in/`, write objects under `demo/out/`, and write logs
to CloudWatch. That is the entire footprint the handler needs, and it is
deliberately narrower than `AdministratorAccess`, so a bug in this one
function can't reach any other bucket or resource. Granting only the
permissions a piece of code needs, and nothing more, is called least
privilege.

In [ ]:
execution_policy = {
    "Version": "2012-10-17",
    "Statement": [
        {
            "Effect": "Allow",
            "Action": ["s3:GetObject"],
            "Resource": [f"arn:aws:s3:::{BUCKET}/in/*"],
        },
        {
            "Effect": "Allow",
            "Action": ["s3:PutObject"],
            "Resource": [f"arn:aws:s3:::{BUCKET}/out/*"],
        },
        {
            "Effect": "Allow",
            "Action": ["logs:CreateLogGroup", "logs:CreateLogStream", "logs:PutLogEvents"],
            "Resource": "arn:aws:logs:*:*:*",
        },
    ],
}

iam.put_role_policy(
    RoleName=ROLE_NAME,
    PolicyName="predict-execution-policy",
    PolicyDocument=json.dumps(execution_policy),
)
print("Policy attached.")

### Package the code

Lambda doesn't run a file directly. It runs a zip archive, with the
handler module sitting at the root of the archive. The cell below zips
the same `handler.py` you just read, in the layout Lambda expects.

In [ ]:
import zipfile

handler_path = Path("demos/01-s3-lambda-notification/lambda_predict/handler.py")
zip_path = Path("demos/01-s3-lambda-notification/predict.zip")

with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
    zf.write(handler_path, "handler.py")

zip_bytes = zip_path.read_bytes()
f"{len(zip_bytes)} bytes"

In [ ]:
try:
    fn = lam.create_function(
        FunctionName=FUNC_NAME,
        Runtime="python3.12",
        Role=role_arn,
        Handler="handler.handler",
        Code={"ZipFile": zip_bytes},
        Timeout=60,
        MemorySize=256,
    )
    func_arn = fn["FunctionArn"]
except lam.exceptions.ResourceConflictException:
    lam.update_function_code(FunctionName=FUNC_NAME, ZipFile=zip_bytes)
    func_arn = lam.get_function(FunctionName=FUNC_NAME)["Configuration"]["FunctionArn"]

func_arn

Lambda functions go through the same lifecycle on Floci as on real AWS:
`Pending` while the platform provisions the runtime container, then
`Active` once it's ready to receive invocations. The loop below polls
until it sees `Active`.

In [ ]:
for _ in range(30):
    conf = lam.get_function_configuration(FunctionName=FUNC_NAME)
    print(conf["State"])
    if conf["State"] == "Active":
        break
    time.sleep(2)

### Create the bucket

The demo bucket holds both the `in/` prefix you'll upload to and the
`out/` prefix `predict` writes to.

In [ ]:
try:
    s3.create_bucket(Bucket=BUCKET)
except s3.exceptions.BucketAlreadyOwnedByYou:
    pass
print(f"Bucket '{BUCKET}' ready.")

### Step 3: the invoke permission

This is a resource policy attached to the Lambda function itself,
separate from the execution role above. It tells Lambda that the S3
service is allowed to invoke it. Skip this step, and S3 refuses to
register the notification you're about to create in step 4. This is the
piece people forget most often, on real AWS as much as here.

In [ ]:
try:
    lam.add_permission(
        FunctionName=FUNC_NAME,
        StatementId="s3invoke",
        Action="lambda:InvokeFunction",
        Principal="s3.amazonaws.com",
        SourceArn=f"arn:aws:s3:::{BUCKET}",
    )
    print("Permission granted.")
except lam.exceptions.ResourceConflictException:
    print("Permission already present.")

### Step 4: the event notification

The last piece: tell S3 that uploads under `in/` should call `predict`.
This is what actually starts the runtime sequence described above.
Everything before this step stayed invisible until now.

In [ ]:
s3.put_bucket_notification_configuration(
    Bucket=BUCKET,
    NotificationConfiguration={
        "LambdaFunctionConfigurations": [
            {
                "LambdaFunctionArn": func_arn,
                "Events": ["s3:ObjectCreated:*"],
                "Filter": {"Key": {"FilterRules": [{"Name": "prefix", "Value": "in/"}]}},
            }
        ]
    },
)
print("Notification configured: in/* -> predict()")

All four steps are done. Before uploading anything, look at what step 4
actually configured, the exact rule S3 will act on:

In [ ]:
s3.get_bucket_notification_configuration(Bucket=BUCKET)["LambdaFunctionConfigurations"]

---
### Upload a file and watch it happen

Everything above was setup. This next cell is the only thing an end user
of this system would ever do: upload a file. Watch what happens in the
next few cells without running any infrastructure call yourself.

In [ ]:
content = b"score\n0.9\n0.2\n0.7\n"

# Clear any leftover output from a previous run of this notebook so the
# timing below reflects this upload, not a stale file.
s3.delete_object(Bucket=BUCKET, Key="out/scores.csv")

upload_start = time.time()
s3.put_object(Bucket=BUCKET, Key="in/scores.csv", Body=content)
print("Uploaded in/scores.csv — watch for the Lambda to pick it up below.")

### Wait for the result

The system you just built is event driven: nothing in it polls for new
files. The loop below only exists so you, reading this notebook, can see
the asynchronous result land instead of taking it on faith. Expect
`out/scores.csv` to appear with a `prediction` column added (1 if
`score` is above 0.5, else 0), usually within a couple of seconds. On a
cold Floci, the first Lambda invocation can take up to 30 seconds while
it pulls the runtime image, so re-run this cell if it times out.

In [ ]:
TIMEOUT_SECONDS = 180
deadline = upload_start + TIMEOUT_SECONDS
result = None

while time.time() < deadline:
    try:
        obj = s3.get_object(Bucket=BUCKET, Key="out/scores.csv")
        result = obj["Body"].read().decode("utf-8")
        break
    except s3.exceptions.NoSuchKey:
        time.sleep(2)

elapsed = time.time() - upload_start
print(f"out/scores.csv appeared after {elapsed:.2f}s\n")
print(result if result else "TIMEOUT — re-run this cell.")

### Why this design holds together

- The `in/` filter is why this doesn't trigger itself in a loop:
  `predict` writes to `out/`, and `out/` doesn't match the trigger.
- The execution policy means a bug in `predict` can't reach anything
  outside `demo/in` and `demo/out`, even though the function itself has
  no idea those limits exist.
- The person uploading and the function processing never call each other
  directly. S3 sits between them, so either side can change without the
  other noticing.

---
## Demo 2: IAM policy enforcement

AWS identities start with nothing. A user with valid credentials and no
attached policy can't do anything at all, not even list buckets, until
someone grants a specific permission. This demo shows that Floci actually
enforces that rule, instead of just recording policies without checking
them, once you turn enforcement on.

```
┌───────┐          ┌─────────────────┐
│ admin │ creates> │ junior + policy │
└───────┘          └─────────────────┘

┌────────┐           ┌─────────────────────┐
│ junior │ attempts> │ s3:ListAllMyBuckets │
└────────┘           └─────────────────────┘
```

You'll create a user called `junior`, watch a request fail with no
policy attached, grant one specific permission, then watch the same
request succeed.

### Turn on enforcement

Open a separate terminal and run:

```bash
make floci-iam-on
```

This restarts Floci with `FLOCI_SERVICES_IAM_ENFORCEMENT_ENABLED=true`. A
cell in this notebook can't do it for you: the container this notebook
runs in has no access to Docker, by design (see the Security section in
the README). If you want to go back to the permissive default afterward,
run `make floci-iam-off`.

Floci drops connections for a few seconds while it restarts. Run the
cell below to confirm it's back before continuing.

In [ ]:
health = urllib.request.urlopen(f"{ENDPOINT}/_floci/health")
json.loads(health.read())["version"]

### Connect as admin

The credentials `test`/`test` bypass enforcement entirely. That's a
Floci convenience for bootstrapping a demo, not something AWS has: real
AWS has no credential that lets you ignore your own account's IAM
policies.

In [ ]:
admin_iam = boto3.client(
    "iam", endpoint_url=ENDPOINT, aws_access_key_id="test", aws_secret_access_key="test"
)
USER = "junior"

### Reset junior

Running this section more than once would otherwise leave old policies
and access keys attached to `junior`. The cell below removes them first,
so the section behaves the same whether this is your first run today or
your fifth.

In [ ]:
try:
    admin_iam.create_user(UserName=USER)
except admin_iam.exceptions.EntityAlreadyExistsException:
    pass

for policy_name in admin_iam.list_user_policies(UserName=USER)["PolicyNames"]:
    admin_iam.delete_user_policy(UserName=USER, PolicyName=policy_name)

for key in admin_iam.list_access_keys(UserName=USER)["AccessKeyMetadata"]:
    admin_iam.delete_access_key(UserName=USER, AccessKeyId=key["AccessKeyId"])

print(f"'{USER}' reset: no policies, no access keys.")

In [ ]:
key = admin_iam.create_access_key(UserName=USER)
junior_access_key = key["AccessKey"]["AccessKeyId"]
junior_secret_key = key["AccessKey"]["SecretAccessKey"]

s3_as_junior = boto3.client(
    "s3",
    endpoint_url=ENDPOINT,
    aws_access_key_id=junior_access_key,
    aws_secret_access_key=junior_secret_key,
)
junior_access_key  # secret key stays in the variable, not printed on screen

### Try without a policy

`junior` now exists with valid credentials, but no policy at all. Run
the next cell and see what AWS does with a request from an identity that
has no permissions.

In [ ]:
try:
    s3_as_junior.list_buckets()
    print("UNEXPECTED SUCCESS — is FLOCI_SERVICES_IAM_ENFORCEMENT_ENABLED=true? "
          "Run `make floci-iam-on` in a terminal, then re-run this cell.")
except ClientError as e:
    status = e.response["ResponseMetadata"]["HTTPStatusCode"]
    code_ = e.response["Error"]["Code"]
    message = e.response["Error"]["Message"]
    print(f"Expected denial observed: {status} {code_}")
    print(message)

### Grant one permission

Attach a policy that allows exactly one action, `s3:ListAllMyBuckets`,
scoped to `Resource: "*"` because bucket listing has no per-bucket scope
in AWS. This grants one named action, not general access.

In [ ]:
allow_list_buckets = {
    "Version": "2012-10-17",
    "Statement": [
        {"Effect": "Allow", "Action": "s3:ListAllMyBuckets", "Resource": "*"}
    ],
}

attach_start = time.time()
admin_iam.put_user_policy(
    UserName=USER, PolicyName="AllowListBuckets", PolicyDocument=json.dumps(allow_list_buckets)
)
print(json.dumps(allow_list_buckets, indent=2))
print("\nThis permits only bucket enumeration — it does not grant reading or")
print("writing any object. junior still can't touch demo/in or demo/out.")

### Try again

Run the same request as before. Expect it to succeed this time, and
almost instantly: on real AWS, a newly attached policy can take a few
seconds to propagate before it takes effect.

In [ ]:
resp = s3_as_junior.list_buckets()
elapsed = time.time() - attach_start
bucket_names = [b["Name"] for b in resp["Buckets"]]
print(f"Success {elapsed:.2f}s after attaching the policy.")
print(f"Buckets: {bucket_names}")
# Non-empty if you ran Demo 1 first in this session (that's expected —
# the point here is the 403 -> success transition, not an empty list).

### What enforcement changes

Floci ships with enforcement off by default. Without
`FLOCI_SERVICES_IAM_ENFORCEMENT_ENABLED=true`, the very first request
above, from `junior` with no policy at all, would also have succeeded.
That's a convenience for getting a demo running quickly, and it's the
opposite of how real AWS behaves: real AWS always enforces IAM.

The policy you attached also took effect immediately here. Real AWS can
take a few seconds to propagate a new policy, so don't build a demo
around an exact timing assumption.

---
## What both demos show

Line the two demos up and one rule explains both: nothing in AWS is
implicit. Every action needs an explicit grant, naming who is allowed to
do it and what resource it applies to.

- Least privilege: `predict`'s execution role reaches exactly
  `demo/in/*` and `demo/out/*`, and nothing else (demo 1, step 2).
- Event-driven decoupling: the person uploading and the function
  processing never call each other directly. The platform wires them
  together through permissions, not through direct calls (demo 1, all
  four steps).
- Deny by default: an identity with valid credentials and no policy can
  do nothing at all (demo 2). Access has to be added; it's never
  assumed.

---
## Cleanup

This doesn't run automatically. When you're done, tear everything down
from a terminal:

```bash
make clean
```